In [1]:
import sys

import pandas as pd

sys.path.append("..")

from src.cleaning import charger_stations, nettoyer_stations

stations_brutes = charger_stations("../data/raw/flux_v2.csv")
print("lignes avant nettoyage :", len(stations_brutes))

# Contrôle n° 1 du CDC : une station décrite deux fois fausserait la jointure EF-06
print("doublons exacts :", stations_brutes.duplicated().sum())
print("id en double :", stations_brutes["id"].duplicated().sum())

# Le flux ne devrait contenir que R et A (ANO-14 ne touche que l'historique)
print(stations_brutes["pop"].value_counts(dropna=False))

# Code postal lu en texte : on vérifie qu'il fait bien 5 caractères partout
print(stations_brutes["Code postal"].str.len().value_counts(dropna=False))

# errors="coerce" transforme en NaN tout ce qui n'est pas un nombre : on peut alors compter ces valeurs
latitude_num = pd.to_numeric(stations_brutes["latitude"], errors="coerce")
longitude_num = pd.to_numeric(stations_brutes["longitude"], errors="coerce")
print("latitude illisible ou vide :", latitude_num.isna().sum())
print("longitude illisible ou vide :", longitude_num.isna().sum())

lignes avant nettoyage : 833
doublons exacts : 0
id en double : 0
pop
R    795
A     38
Name: count, dtype: int64
Code postal
5    833
Name: count, dtype: int64
latitude illisible ou vide : 0
longitude illisible ou vide : 0


In [2]:
stations_propres = nettoyer_stations(stations_brutes)
print(stations_propres.columns.tolist())

# Créer une nouvelle table ne doit faire perdre aucune ligne
assert len(stations_propres) == len(stations_brutes)

['id_station', 'code_postal', 'departement', 'ville', 'implantation', 'latitude', 'longitude']


In [3]:
# errors="coerce" a pu créer des manquants : on les recompte (cours p. 31)
print(stations_propres[["latitude", "longitude"]].isna().sum())
print(stations_propres[["latitude", "longitude"]].describe())

# Bornes de l'Île-de-France fixées dans ANO-11
hors_bornes = stations_propres[
    (stations_propres["latitude"] < 48) | (stations_propres["latitude"] > 49.2)
    | (stations_propres["longitude"] < 1.4) | (stations_propres["longitude"] > 3.6)
]
print("stations hors bornes :", len(hors_bornes))
hors_bornes

latitude     0
longitude    0
dtype: int64
         latitude   longitude
count  833.000000  833.000000
mean    48.809373    2.370399
std      0.176814    0.300642
min     48.145000    1.489000
25%     48.718000    2.198000
50%     48.837000    2.359273
75%     48.935000    2.527000
max     49.157000    3.361000
stations hors bornes : 0


,id_station,code_postal,departement,ville,implantation,latitude,longitude
